# RenoMed-CKD

## Chronic Kidney Disease Prediction Using Machine Learning

### Project Overview
Chronic Kidney Disease (CKD) is a serious global health condition that affects kidney function and can lead to kidney failure if not detected early. Early prediction and diagnosis are important for improving patient outcomes and reducing complications.

This project aims to build a machine learning system capable of predicting the likelihood of CKD using patient clinical and laboratory data.

### Objectives
The main goals of this project are to:

- Perform data cleaning and preprocessing on a healthcare dataset
- Handle missing and inconsistent clinical records
- Explore relationships between medical features and CKD
- Train and compare Random Forest and Linear Regression machine learning models
- Evaluate model performance
- Identify important clinical predictors of CKD by analyzing features (over 23 in the dataset)
- Build a foundation for future healthcare AI applications

### Dataset Information
The dataset contains patient clinical measurements such as:

- Age
- Blood Pressure
- Albumin
- Sugar
- Blood Glucose Random
- Blood Urea
- Serum Creatinine
- Sodium
- Potassium
- Hemoglobin
- White Blood Cell Count

### Target Variable
- CKD
- Not CKD

### Why This Project Matters
Healthcare datasets are often messy and incomplete, making preprocessing an important part of the machine learning pipeline. This project focuses not only on prediction performance, but also on understanding clinically relevant features associated with Chronic Kidney Disease.

In healthcare applications, identifying high-risk patients early can contribute to timely medical intervention and improved patient care.

---

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


---

## Data Loading

The dataset `kidney_disease.csv` has been uploaded and contains 400 rows with various patient clinical and laboratory data. The target variable is 'classification', indicating 'ckd' or 'notckd'.

In [2]:
df = pd.read_csv('/content/kidney_disease.csv')

print(f"Dataset loaded successfully. It contains {df.shape[0]} rows and {df.shape[1]} columns.")
print("First 5 rows of the dataset:")
display(df.head())

Dataset loaded successfully. It contains 400 rows and 26 columns.
First 5 rows of the dataset:


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


---

## Data Cleaning and Preprocessing

Healthcare datasets often contain missing values and inconsistent data types, which need to be addressed before machine learning. This dataset also has NaNs, and numeric features need to be converted to floats. As instructed by the source, all rows with even a single NaN value will be deleted.

In [4]:
# Initial check for missing values
print("Missing values before cleaning:")
display(df.isnull().sum()[df.isnull().sum() > 0])

# Drop rows with any NaN values
original_rows = df.shape[0]
df.dropna(inplace=True)
cleaned_rows = df.shape[0]

print(f"\nDropped {original_rows - cleaned_rows} rows with NaN values. Remaining rows: {cleaned_rows}")

# Verify no more missing values
print("\nMissing values after dropping rows:")
display(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before cleaning:


,0



Dropped 0 rows with NaN values. Remaining rows: 158

Missing values after dropping rows:


,0


### Handling Mixed Data Types and New NaNs

Some columns in healthcare datasets might be loaded as 'object' (string) type even if they are intended to be numeric. This can happen if there are non-numeric characters (like '?' or spaces) within the numeric data.

To address this, I'll use `pd.to_numeric()`:
- It attempts to convert the values in the specified columns to a numeric type (like float).
- The `errors='coerce'` argument is crucial here: if `pd.to_numeric()` encounters any value that *cannot* be converted into a number, instead of raising an error, it will convert that problematic value into `NaN` (Not a Number).

After this conversion, new `NaN` values might be introduced in rows where previously non-numeric entries existed. Following the instruction to remove *all* rows with `NaNs`, I performed `df.dropna(inplace=True)` again to get rid of these newly introduced missing values. As noted, I started with 400 rows, dropped 242 initially, leaving 158 rows before this step. This ensures my dataset remains clean and purely numeric for subsequent analysis.

In [5]:
# Identify columns that should be numeric but might be objects due to non-numeric entries
# Based on typical CKD datasets, these columns are likely numeric:
# 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane'

# Let's check current data types
print("\nData types before conversion:")
display(df.info())

# Columns that are expected to be numeric but might be object type due to '?' or other non-numeric strings
# We'll explicitly list them and convert using errors='coerce'
numeric_cols = [
    'age', 'bp', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# After coercing, there might be new NaNs, so we drop them again as per the instruction
original_rows_after_coerce = df.shape[0]
df.dropna(inplace=True)
cleaned_rows_after_coerce = df.shape[0]

print(f"\nDropped {original_rows_after_coerce - cleaned_rows_after_coerce} rows introduced by numeric coercion.")

print("\nData types after numeric conversion and dropping new NaNs:")
display(df.info())

print("First 5 rows of the cleaned dataset:")
display(df.head())


Data types before conversion:
<class 'pandas.core.frame.DataFrame'>
Index: 158 entries, 3 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              158 non-null    int64  
 1   age             158 non-null    float64
 2   bp              158 non-null    float64
 3   sg              158 non-null    float64
 4   al              158 non-null    float64
 5   su              158 non-null    float64
 6   rbc             158 non-null    object 
 7   pc              158 non-null    object 
 8   pcc             158 non-null    object 
 9   ba              158 non-null    object 
 10  bgr             158 non-null    float64
 11  bu              158 non-null    float64
 12  sc              158 non-null    float64
 13  sod             158 non-null    float64
 14  pot             158 non-null    float64
 15  hemo            158 non-null    float64
 16  pcv             158 non-null    object 
 17  wc       

None


Dropped 0 rows introduced by numeric coercion.

Data types after numeric conversion and dropping new NaNs:
<class 'pandas.core.frame.DataFrame'>
Index: 158 entries, 3 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              158 non-null    int64  
 1   age             158 non-null    float64
 2   bp              158 non-null    float64
 3   sg              158 non-null    float64
 4   al              158 non-null    float64
 5   su              158 non-null    float64
 6   rbc             158 non-null    object 
 7   pc              158 non-null    object 
 8   pcc             158 non-null    object 
 9   ba              158 non-null    object 
 10  bgr             158 non-null    float64
 11  bu              158 non-null    float64
 12  sc              158 non-null    float64
 13  sod             158 non-null    float64
 14  pot             158 non-null    float64
 15  hemo            158 no

None

First 5 rows of the cleaned dataset:


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
9,9,53.0,90.0,1.020,2.0,0.0,abnormal,abnormal,present,notpresent,...,29,12100,3.7,yes,yes,no,poor,no,yes,ckd
11,11,63.0,70.0,1.010,3.0,0.0,abnormal,abnormal,present,notpresent,...,32,4500,3.8,yes,yes,no,poor,yes,no,ckd
14,14,68.0,80.0,1.010,3.0,2.0,normal,abnormal,present,present,...,16,11000,2.6,yes,yes,yes,poor,yes,no,ckd
20,20,61.0,80.0,1.015,2.0,0.0,abnormal,abnormal,notpresent,notpresent,...,24,9200,3.2,yes,yes,yes,poor,yes,yes,ckd


### Output Summary: Data Type Conversion and Row Count

From the output of the previous cell, we observe the following:

1.  **Initial Data Types (before `pd.to_numeric`):** The dataset with 158 rows showed several columns, such as `pcv`, `wc`, and `rc`, as `object` data types, despite being inherently numeric. Other categorical features like `rbc`, `pc`, `htn`, etc., were also `object` type.

2.  **Numeric Coercion Result:** The `pd.to_numeric` operation successfully converted `pcv` to `int64`, `wc` to `int64`, and `rc` to `float64`. This confirms these columns are now correctly represented as numeric values.

3.  **Row Count after Coercion:** Crucially, the message "Dropped 0 rows introduced by numeric coercion" indicates that no new `NaN` values were generated during this conversion process in the remaining 158 rows. This means all values in the targeted numeric columns were successfully converted without encountering unresolvable non-numeric entries. The dataset retains its 158 rows.

4.  **Final Data Types:** After the conversions, the DataFrame now has 12 `float64`, 3 `int64`, and 11 `object` columns. The `object` columns are primarily the categorical features that will require further encoding.

I've successfully handled missing values and ensured the numerical features are correctly typed as `float` or `int`, I still have several columns that are `object` data types:

*   `rbc`, `pc`, `pcc`, `ba`, `htn`, `dm`, `cad`, `appet`, `pe`, `ane` (These are likely **categorical features** with text values like 'present'/'absent' or 'yes'/'no').
*   `classification` (This is my **target variable**, which also contains text 'ckd'/'notckd').

These categorical columns need further processing, typically through **encoding techniques** (like one-hot encoding or label encoding), to convert their text values into numerical representations that machine learning algorithms can understand. I'll address this in the next steps.